In [1]:
import re
import requests
from collections import defaultdict

<h5>Necessary imports: regular expressions, requests and the defaultdict data structure</h5>

In [2]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

DOC_TITLES = [
    "Pizza",
    "The Hitchhiker's Guide to the Galaxy",
    "George Gershwin",
    "Paul Mccartney",
    "Cheese"
]


session = requests.Session()

HEADERS = {
    "User-Agent": "Python/requests"
}

<h5>Constants defenition - Wikipedia's API, page titles and request necessaties</h5>

In [3]:
def wiki(title):
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": False,
        "titles": title,
        "format": "json",
        "redirects": 0,
        "formatversion": 2
    }
    resp = session.get(WIKI_API, params=params, headers=HEADERS).json()
    pages = resp.get("query", {}).get("pages", [])
    if not pages:
        return ""
    return pages[0].get("extract", "")

<h5>A function to handle and execute calls to Wikipedia's API</h5>

In [4]:
def build_inverted_index(docs):
    inverted = defaultdict(list)
    for doc_id, text in docs.items():
        tokens = [t for t in (re.split(r"\W+", text.lower())) if t]
        uniqueterms = set(tokens)
        for term in uniqueterms:
            posting = {"doc_id": doc_id}
            inverted[term].append(posting)
    return dict(inverted)

<h5>A function that tokenizes and indexes every word in each page in accordance to BoW, each collection of tokens sourced from one document is stored in a set, the sets are then added to a dictionary where each unique token is stored along with its corresponding document of origin</h5>

In [8]:
docs = {}
for i, title in enumerate(DOC_TITLES, start=1):
    print(f"Fetching: {title}")
    text = wiki(title)
    docs[i] = {"title": title, "text": text}
doc_texts = {doc_id: info["text"] for doc_id, info in docs.items()}
index = build_inverted_index(doc_texts)
sample_terms = input('input terms to search ').split(', ')
for term in sample_terms:
    postings = index.get(term.lower(), [])
    print(f"{len(postings)} postings found for {term}")
    for p in postings:
        title = docs[p["doc_id"]]["title"]
        print(f"  doc_id={p['doc_id']} title='{title}'")

Fetching: Pizza
Fetching: The Hitchhiker's Guide to the Galaxy
Fetching: George Gershwin
Fetching: Paul Mccartney
Fetching: Cheese


input terms to search  music, life, beatles, popular


3 postings found for music
  doc_id=2 title='The Hitchhiker's Guide to the Galaxy'
  doc_id=3 title='George Gershwin'
  doc_id=4 title='Paul Mccartney'
5 postings found for life
  doc_id=1 title='Pizza'
  doc_id=2 title='The Hitchhiker's Guide to the Galaxy'
  doc_id=3 title='George Gershwin'
  doc_id=4 title='Paul Mccartney'
  doc_id=5 title='Cheese'
1 postings found for beatles
  doc_id=4 title='Paul Mccartney'
5 postings found for popular
  doc_id=1 title='Pizza'
  doc_id=2 title='The Hitchhiker's Guide to the Galaxy'
  doc_id=3 title='George Gershwin'
  doc_id=4 title='Paul Mccartney'
  doc_id=5 title='Cheese'
